<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<font size="5" color='grey'> <b> Chat Memory Patterns — Python-Implementierung</b></font> </br>

---

In [1]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt
)
setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.15
langchain-chroma                         1.1.0
langchain-classic                        1.0.4
langchain-community                      0.4.1
langchain-core                           1.3.1
langchain-ollama                         1.1.0
langchain-openai                         1.2.0
langchain-text-splitters                 1.1.2
langgraph                                1.1.9
langgraph-checkpoint                     4.1.0
langgraph-checkpoint-sqlite              3.1.0
langgraph-prebuilt                       1.0.10
langgraph-sdk                            0.3.13

IP-Adresse: 34.63.226.21
Hostname: 21.226.63.34.bc.googleusercontent.com
Stadt: Council Bluffs
Region: Iowa
Land: US
Koordinaten: 41.2619,-95.8608
Provider: AS396982 Google LLC
Postleitzahl: 51502
Zeitzone: America/Chicago


# 1 | Intro
---

<p><font color='black' size="5">
Zustandslosigkeit von LLMs
</font></p>

Large Language Models (LLMs) wie GPT sind von Natur aus **zustandslos** — sie verfügen über kein eingebautes Gedächtnis. Jede Anfrage wird isoliert verarbeitet, ohne Bezug zu vorherigen Interaktionen. Deshalb muss der Chatverlauf bei jeder Anfrage neu übergeben werden.

```
Ohne Memory:
User: "Mein Name ist Max"
AI: "Hallo Max!"
User: "Wie heisse ich?"
AI: "Das habe ich nicht gespeichert."
```

**Gängige Memory-Patterns:**

| Pattern | Beschreibung | Anwendungsfall |
|---------|--------------|----------------|
| **Python-Liste** | Einfachste Lösung | Prototyping, kurze Sessions |
| **Trimming** | Nur letzte N Nachrichten an LLM | Token-Limit einhalten |
| **Summary** | Alte Nachrichten zusammenfassen | Lange Sessions, Kontext erhalten |
| **SQLite direkt** | Persistente Speicherung | Production, Multi-User |

> **📌 Hinweis:** Dieses Notebook zeigt Memory-Patterns in **reinem Python** (Listen, Dicts, SQLite-Modul) — ohne LangGraph-Framework. Für die LangGraph-Variante derselben Patterns (StateGraph + Checkpointer) gibt es ein separates Notebook: `M07_Chat_Memory_Patterns_stategraph.ipynb`.

In [35]:
#@markdown Chat Memory Patterns — Python-Implementierung
diagram = """
%%{init: {'theme':'forest'}}%%
graph TD
    root[Chat Memory Patterns]

    root -->|Short-term / Manuell| manual[Python-Liste]
    manual --> manual_desc["Einfache Liste im RAM<br/>Prototyping, kurze Sessions"]

    root -->|Session-Mgmt / Python-Dict| pydict["sessions = {}"]

    pydict -->|Context Management| strat[Strategien]

    strat --> trim[Trimming]
    trim --> trim_desc["Sliding Window<br/>letzte N Msgs ans LLM"]

    strat --> summ[Summary]
    summ --> summ_desc["LLM fasst zusammen<br/>Summary separat gespeichert"]

    pydict -->|persistentes Memory| persist[Persistenz]

    persist --> sqlite[SQLite direkt]
    sqlite --> sqlite_desc["sqlite3-Modul<br/>Multi-Thread, persistent"]

    style root fill:#2ecc71,stroke:#27ae60,stroke-width:2px,color:white
    style pydict fill:#3498db,stroke:#2980b9,stroke-width:2px,color:white
    style manual fill:#95a5a6,stroke:#7f8c8d,stroke-width:2px,color:white
    style strat fill:#f1c40f,stroke:#f39c12,color:black
    style persist fill:#e67e22,stroke:#d35400,color:white
"""
mermaid(diagram, width=800)

# 2 | Short-term Memory (Python-Liste)
---

Die einfachste Form von Memory: Eine **Python-Liste**, die alle Nachrichten speichert und bei jedem API-Call mitgesendet wird.

In [3]:
# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ── Modell ────────────────────────────────────────────────────────────────────
from genai_lib.model_config import BASELINE

In [4]:
# System-Prompt
system_prompt = "Du bist ein hilfreicher und humorvoller KI-Assistent."

# Prompt-Template mit Historie (MessagesPlaceholder nimmt die Historie entgegen)
prompt = ChatPromptTemplate.from_messages([
    ("system", "{system_prompt}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_input}")
])

# Modell
llm = init_chat_model(BASELINE)

# Parser
parser = StrOutputParser()

# Chain
chain = prompt | llm | parser

In [5]:
# Chat-Funktion mit manueller Historien-Verwaltung
def chat(chat_history, user_input):
    """Fuhrt eine Chat-Interaktion mit manueller Historien-Verwaltung durch."""

    # Chain aufrufen (Historie wird im Prompt mitgeschickt)
    response = chain.invoke({
        'system_prompt': system_prompt,
        'chat_history': chat_history,
        'user_input': user_input
    })

    # Ausgabe
    mprint(f"### 🧑‍🦱 Mensch:\n{user_input}")
    mprint(f"### 🤖 KI:\n{response}\n")

    # Memory (Liste) MANUELL aktualisieren
    chat_history.extend([HumanMessage(content=user_input), AIMessage(content=response)])

    return chat_history

In [6]:
# Historie initialisieren
chat_history = [SystemMessage(content=system_prompt)]

# Konversation
chat_history = chat(chat_history, "Mein Name ist Max")
chat_history = chat(chat_history, "Ich mag Python-Programmierung")
chat_history = chat(chat_history, "Weisst du noch, wie ich heisse und was ich mag?")

### 🧑‍🦱 Mensch:
Mein Name ist Max

### 🤖 KI:
Hey Max! 👋  
Willkommen an Bord. Wie kann ich dir helfen—geht’s um Mathe, einen Text, Programmieren oder was komplett anderes (z.B. „KI, gib mir eine witzige Idee für ein Geschenk“)? 😄


### 🧑‍🦱 Mensch:
Ich mag Python-Programmierung

### 🤖 KI:
Nice, Max! Python ist wie ein Schweizer Taschenmesser für Gedanken—nur mit weniger Schraubenzieher und mehr „import“. 😄🐍

Womit willst du anfangen?

1) **Einsteiger-Setup** (Programme ausführen, virtuelle Umgebung, Basics)  
2) **Erstes kleines Projekt** (z.B. Rechner, To-do, Web-Scraper-light)  
3) **Daten/Automatisierung** (CSV/Excel, Dateiverarbeitung)  
4) **Web** (Flask/FastAPI)  
5) **Programmieren üben** (Übungsaufgaben + Feedback)  

Sag mir kurz: Was ist dein Level (Anfänger / bisschen / fortgeschritten) und was reizt dich am meisten?


### 🧑‍🦱 Mensch:
Weisst du noch, wie ich heisse und was ich mag?

### 🤖 KI:
Ja 😄  
Du heißt **Max** und du magst **Python-Programmierung**.


In [7]:
mprint("### Gespeicherte Nachrichten (Liste):\n---")
for msg in chat_history:
    mprint(f"  **{msg.type}**:   {msg.content}")

### Gespeicherte Nachrichten (Liste):
---

  **system**:   Du bist ein hilfreicher und humorvoller KI-Assistent.

  **human**:   Mein Name ist Max

  **ai**:   Hey Max! 👋  
Willkommen an Bord. Wie kann ich dir helfen—geht’s um Mathe, einen Text, Programmieren oder was komplett anderes (z.B. „KI, gib mir eine witzige Idee für ein Geschenk“)? 😄

  **human**:   Ich mag Python-Programmierung

  **ai**:   Nice, Max! Python ist wie ein Schweizer Taschenmesser für Gedanken—nur mit weniger Schraubenzieher und mehr „import“. 😄🐍

Womit willst du anfangen?

1) **Einsteiger-Setup** (Programme ausführen, virtuelle Umgebung, Basics)  
2) **Erstes kleines Projekt** (z.B. Rechner, To-do, Web-Scraper-light)  
3) **Daten/Automatisierung** (CSV/Excel, Dateiverarbeitung)  
4) **Web** (Flask/FastAPI)  
5) **Programmieren üben** (Übungsaufgaben + Feedback)  

Sag mir kurz: Was ist dein Level (Anfänger / bisschen / fortgeschritten) und was reizt dich am meisten?

  **human**:   Weisst du noch, wie ich heisse und was ich mag?

  **ai**:   Ja 😄  
Du heißt **Max** und du magst **Python-Programmierung**.

<p><font color='darkblue' size="4">
Problem:
</font></p>

- Keine automatische Session-Verwaltung (Multi-User)
- Manuelles Memory-Management fehleranfallig
- **Bei langen Konversationen: Token-Limit wird uberschritten!**

# 3 | Python-basiertes Session-Management
---

Die Grundidee ist ein **Python-Dictionary**, dessen Keys die Thread-IDs sind. Jeder Wert ist eine **Liste** von Nachrichten — die vollständige Interaktion zwischen Mensch und Sprachmodell für diesen Thread:

```python
sessions = {
    "max": [
        HumanMessage("Mein Name ist Max"),   # 1. Listenelement
        AIMessage("Hallo Max!"),             # 2. Listenelement
        HumanMessage("Ich komme aus München"),
        AIMessage("Schön, aus München zu hören!"),
    ],
    "emma": [
        HumanMessage("Hi, ich bin Emma"),    # 1. Listenelement
        AIMessage("Hallo Emma!"),            # 2. Listenelement
    ],
}
# sessions["max"]  → 4 Listenelemente
# sessions["emma"] → 2 Listenelemente
```

Dieses Muster zieht sich durch alle drei Unterabschnitte (3.1–3.3). Trimming und Summary ändern nur, **welcher Ausschnitt der Liste ans LLM gesendet wird** — das Dictionary-Schema bleibt identisch.

<p><font color='black' size="5">
3.1 | Basismodell
</font></p>

---

In [8]:
#@markdown Funktionsaufrufe Basismodell
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef dictStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;

    subgraph UI ["1. Input / Output"]
        direction TB
        Start["`ask(thread_id, user_input)`"]:::loopStyle
        Ende["`mprint() / Ausgabe`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    subgraph Engine ["2. Python-Session-Verwaltung"]
        direction TB
        Load["`1. sessions[thread_id]`"]:::dictStyle
        LLM["`2. llm.invoke(context)`"]:::dictStyle
        Save["`3. append(HumanMsg + AIMsg)`"]:::dictStyle
        Load --> LLM --> Save
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    subgraph Storage ["3. Speicher"]
        Dict["`sessions = {}<br/>Python-Dict im RAM`"]:::saveStyle
    end
    style Storage fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    Start --> Load
    Load <--> Dict
    Save --> Dict
    Save --> Ende
"""
mermaid(diagram, width=1000)

> **📌 Hinweis:** `llm` und `system_prompt` wurden in Abschnitt 2 definiert (Zellen „Modell" und „System-Prompt") und stehen hier als globale Variablen zur Verfügung.

In [9]:
# Session-Verwaltung mit Python-Dictionary
sessions = {}  # thread_id -> list[BaseMessage]

def ask(thread_id: str, user_input: str) -> str:
    if thread_id not in sessions:
        sessions[thread_id] = []

    sessions[thread_id].append(HumanMessage(content=user_input))
    context  = [SystemMessage(content=system_prompt)] + sessions[thread_id]
    response = llm.invoke(context)
    sessions[thread_id].append(response)

    mprint(f"**[{thread_id}] Mensch:** {user_input}")
    mprint(f"**[{thread_id}] KI:** {response.content}\n")
    return response.content

In [10]:
mprint("## Basismodell Demo")
mprint("---")

for thread_id, user_input in [
    ("max",  "Hallo! Ich bin Max aus München."),
    ("max",  "Ich programmiere gerne in Python."),
    ("emma", "Hi! Ich bin Emma und mag Machine Learning."),
    ("max",  "Woher komme ich und was ist mein Hobby?"),
]:
    ask(thread_id, user_input)

mprint("### Gespeicherte Sessions")
for tid in ["max", "emma"]:
    mprint(f"- **{tid}**: {len(sessions[tid])} Nachrichten")

mprint("### Historie: max")
for idx, msg in enumerate(sessions["max"], 1):
    role = "Mensch" if msg.type == "human" else "KI"
    mprint(f"{idx}. **{role}:** {msg.content}")

## Basismodell Demo

---

**[max] Mensch:** Hallo! Ich bin Max aus München.

**[max] KI:** Hallo Max! 😄 Willkommen bei mir!  
München grüßt zurück — wie läuft’s so? Bist du eher auf der Suche nach was für den Alltag (Tipps/Planung), oder willst du gerade was Bestimmtes lösen/erfinden/entscheiden?


**[max] Mensch:** Ich programmiere gerne in Python.

...

2. **KI:** Hallo Max! 😄 Willkommen bei mir!  
München grüßt zurück — wie läuft’s so? Bist du eher auf der Suche nach was für den Alltag (Tipps/Planung), oder willst du gerade was Bestimmtes lösen/erfinden/entscheiden?

3. **Mensch:** Ich programmiere gerne in Python.

4. **KI:** Nice, Max! Python ist einfach der Hausschlüssel fürs Universum 😄🔑🐍

Wobei kann ich dir helfen? Z.B.:  
- Debugging (Fehlermeldung/Code schicken)  
- Skript schreiben/optimieren  
- Web/Flask/FastAPI  
- Datenanalyse (pandas)  
- Automatisierung/Scraping  
- Algorithmen/Übungen (LeetCode, etc.)

Sag mir kurz: Was genau baust du gerade (und in welchem Projekt/Framework)?

5. **Mensch:** Woher komme ich und was ist mein Hobby?

6. **KI:** Du kommst aus **München** (du hast es mir ja gesagt 😊).

Zu deinem Hobby: Das, was ich bisher sicher weiß, ist **dass du gerne in Python programmierst** — das würde ich als dein Hobby einstufen. 🐍

Wenn du willst, erzähl mir kurz noch 1–2 Dinge über dich, dann kann ich’s genauer formulieren.

<p><font color='black' size="5">
3.2 | Trimming (Sliding Window)
</font></p>

---

**Problem:** Bei langen Konversationen wächst `sessions[thread_id]` unbegrenzt — das Token-Limit des LLM wird überschritten.

**Lösung:** Beim LLM-Aufruf werden nur die letzten `MAX_MESSAGES` Nachrichten übergeben. Die **vollständige Liste** bleibt im `sessions`-Dict erhalten — nur das LLM-Kontextfenster wird begrenzt.

In [11]:
#@markdown Funktionsaufrufe Trimming
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef dictStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;

    subgraph UI ["1. Input / Output"]
        Start["`ask_trimmed(thread_id, user_input)`"]:::loopStyle
        Ende["`mprint() / Ausgabe`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    subgraph Engine ["2. Trimming-Logik"]
        direction TB
        Load["`1. Vollständige Liste laden`"]:::dictStyle
        Trim["`2. messages[-MAX_MESSAGES:]`"]:::dictStyle
        LLM["`3. llm.invoke(SystemMsg + window)`"]:::dictStyle
        Save["`4. Vollständige Liste speichern`"]:::dictStyle
        Load --> Trim --> LLM --> Save
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    subgraph Cfg ["3. Konfiguration"]
        C["`MAX_MESSAGES = 6`"]:::saveStyle
    end
    style Cfg fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    Start --> Load
    Trim -.->|nutzt| C
    Save --> Ende
"""
mermaid(diagram, width=1000)

In [ ]:
MAX_MESSAGES = 6
trimmed_sessions = {}  # thread_id -> list[BaseMessage]

def ask_trimmed(thread_id: str, user_input: str) -> str:
    if thread_id not in trimmed_sessions:
        trimmed_sessions[thread_id] = []

    trimmed_sessions[thread_id].append(HumanMessage(content=user_input))

    # Sliding window: nur die letzten MAX_MESSAGES Nachrichten ans LLM senden.
    # Die vollständige Liste (trimmed_sessions) bleibt unverändert erhalten —
    # nur das LLM-Kontextfenster wird begrenzt, nicht der gespeicherte Verlauf.
    window   = trimmed_sessions[thread_id][-MAX_MESSAGES:]
    context  = [SystemMessage(content=system_prompt)] + window
    response = llm.invoke(context)

    trimmed_sessions[thread_id].append(response)
    mprint(f"**Mensch:** {user_input}")
    mprint(f"**KI:** {response.content}\n")
    return response.content

In [13]:
mprint("## Trimming Demo (max. 6 Nachrichten im LLM-Kontext)")
mprint("---")

for nr, user_input in enumerate([
    "Mein Geheimcode lautet: BLAU-42.",
    "Ich komme aus München.",
    "Ich mag Python.",
    "Ich arbeite als Data Scientist.",
    "Was war mein Geheimcode?",       # fällt aus dem Trimming-Fenster
], 1):
    ask_trimmed("trim_test", user_input)

## Trimming Demo (max. 6 Nachrichten im LLM-Kontext)

---

**Mensch:** Mein Geheimcode lautet: BLAU-42.

**KI:** Oh cool—dein Geheimcode ist also **BLAU-42** 😄  
Soll ich damit irgendwas Bestimmtes machen (z. B. in eine Liste eintragen, als Passwort-Placeholder verwenden, oder in ein Rätsel einbauen)?


**Mensch:** Ich komme aus München.

...

**KI:** Sehr gut, **Python** ist ein Klassiker—leicht lesbar, stark in der Praxis und macht sogar manchmal Spaß beim Debuggen 😄

Womit machst du’s am liebsten?
- Automatisierung / Scripting  
- Web (z. B. Flask/FastAPI)  
- Datenanalyse (pandas)  
- ML / Deep Learning  
- Gaming / Bots  
- „Ich bastle einfach rum“  

Wenn du magst, sag mir auch dein aktuelles Projekt—dann kann ich dir gezielt helfen (oder dir ein kleines Code-Snippet basteln).


**Mensch:** Ich arbeite als Data Scientist.

**KI:** Geil—Data Science in München klingt nach **„Cappuccino + Confusion Matrix“** 😄

Woran arbeitest du gerade am meisten?
- klassische ML-Modelle (XGBoost/LightGBM, Random Forest, …)
- Deep Learning (PyTorch/TensorFlow)
- Time Series / Forecasting
- NLP / LLMs
- Datenpipeline/Feature Engineering
- Experimentieren & Evaluieren (CV, Leakage, Metriken)

Wenn du willst, sag mir kurz:
1) dein Datentyp (tabellarisch, text, time series, …)  
2) Ziel (klassifizieren, regression, forecasting, …)  
3) was gerade hakt (Overfitting, Metrik schlecht, Latenz, Datenqualität, …)  

Dann helf ich dir mit einem konkreten Plan oder Code.


**Mensch:** Was war mein Geheimcode?

**KI:** Ich hab keinen Geheimcode von dir bekommen—zumindest nicht in unserem Chat. 😄  
Hast du den Code irgendwo gepostet (oder meinst du einen aus einer früheren Unterhaltung)?

Wenn du mir den Hinweis gibst (z. B. „kommt aus deinem ersten Satz“ oder „ist 4 Wörter lang“), kann ich mit dir zusammen den plausiblen Code rekonstruieren.


In [14]:
# Vollständige Liste vs. LLM-Fenster visualisieren
all_msgs = trimmed_sessions["trim_test"]
cutoff   = len(all_msgs) - MAX_MESSAGES

mprint("### Vollständige Historie vs. LLM-Fenster")
mprint(f"{len(all_msgs)} Nachrichten gespeichert · LLM sah zuletzt {min(MAX_MESSAGES, len(all_msgs))}")
mprint("---")
for idx, msg in enumerate(all_msgs, 1):
    role   = "Mensch" if msg.type == "human" else "KI"
    marker = "✅" if idx > cutoff else "❌"
    mprint(f"{marker} {idx}. **{role}:** {msg.content}")
mprint("---")
mprint("**Legende:** ✅ im Fenster (LLM sah diese) · ❌ ausgeblendet (aber gespeichert!)")

### Vollständige Historie vs. LLM-Fenster

10 Nachrichten gespeichert · LLM sah zuletzt 6

---

❌ 1. **Mensch:** Mein Geheimcode lautet: BLAU-42.

❌ 2. **KI:** Oh cool—dein Geheimcode ist also **BLAU-42** 😄  
Soll ich damit irgendwas Bestimmtes machen (z. B. in eine Liste eintragen, als Passwort-Placeholder verwenden, oder in ein Rätsel einbauen)?

...

✅ 8. **KI:** Geil—Data Science in München klingt nach **„Cappuccino + Confusion Matrix“** 😄

Woran arbeitest du gerade am meisten?
- klassische ML-Modelle (XGBoost/LightGBM, Random Forest, …)
- Deep Learning (PyTorch/TensorFlow)
- Time Series / Forecasting
- NLP / LLMs
- Datenpipeline/Feature Engineering
- Experimentieren & Evaluieren (CV, Leakage, Metriken)

Wenn du willst, sag mir kurz:
1) dein Datentyp (tabellarisch, text, time series, …)  
2) Ziel (klassifizieren, regression, forecasting, …)  
3) was gerade hakt (Overfitting, Metrik schlecht, Latenz, Datenqualität, …)  

Dann helf ich dir mit einem konkreten Plan oder Code.

✅ 9. **Mensch:** Was war mein Geheimcode?

✅ 10. **KI:** Ich hab keinen Geheimcode von dir bekommen—zumindest nicht in unserem Chat. 😄  
Hast du den Code irgendwo gepostet (oder meinst du einen aus einer früheren Unterhaltung)?

Wenn du mir den Hinweis gibst (z. B. „kommt aus deinem ersten Satz“ oder „ist 4 Wörter lang“), kann ich mit dir zusammen den plausiblen Code rekonstruieren.

---

**Legende:** ✅ im Fenster (LLM sah diese) · ❌ ausgeblendet (aber gespeichert!)

<p><font color='black' size="5">
3.3 | Summary (LLM-basierte Zusammenfassung)
</font></p>

---

**Problem:** Trimming verwirft alte Informationen vollständig — wichtiger Kontext geht verloren.

**Alternative:** **Summarization** — ein LLM fasst alte Nachrichten zusammen.

Das Dict-Listen-Schema bleibt identisch zu 3.1 und 3.2 — nur die Nachrichten-Liste wird periodisch gekürzt. Die Zusammenfassung lebt in einem **zweiten, flachen Dictionary** (`summaries`), getrennt von den Nachrichten:

```python
summary_sessions = {}   # thread_id -> list[BaseMessage]   (aktuelle Nachrichten)
summaries        = {}   # thread_id -> str                  (Zusammenfassung als Text)
```

Der LLM-Kontext wird immer korrekt aufgebaut: `[system_prompt, summary, aktuelle_nachrichten]`

**Strategie:**
1. Wenn `len(summary_sessions[thread_id]) > SUMMARY_THRESHOLD`: alte Nachrichten zusammenfassen
2. Neue Summary enthält alte Summary + neue Fakten (Rolling Summary)
3. `summary_sessions[thread_id]` auf die letzten `KEEP_RECENT` Elemente kürzen
4. LLM-Kontext: System → Summary → aktuelle Nachrichten

In [15]:
#@markdown Funktionsaufrufe Summary
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef dictStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;
    classDef decStyle fill:#fce4ec,stroke:#e91e63,stroke-width:2px;

    subgraph UI ["1. Input / Output"]
        Start["`ask_summary(thread_id, user_input)`"]:::loopStyle
        Ende["`mprint() / Ausgabe`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    subgraph Engine ["2. Summary-Logik"]
        direction TB
        Load["`1. State laden<br/>messages + summary`"]:::dictStyle
        Check{"`len > THRESHOLD?`"}:::decStyle
        Sum["`2a. summarize()<br/>LLM: old_summary + to_summarize`"]:::dictStyle
        Trim["`2b. messages = messages[-KEEP_RECENT:]`"]:::dictStyle
        Build["`3. Kontext aufbauen<br/>[system, summary, recent]`"]:::dictStyle
        LLM["`4. llm.invoke(context)`"]:::dictStyle
        Load --> Check
        Check -->|Ja| Sum --> Trim --> Build --> LLM
        Check -->|Nein| Build
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    subgraph SumCfg ["3. Konfiguration"]
        direction TB
        Cfg["`SUMMARY_THRESHOLD = 8<br/>KEEP_RECENT = 4`"]:::saveStyle
        State["`summary_sessions = {}<br/>messages + summary getrennt`"]:::saveStyle
    end
    style SumCfg fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    Start --> Load
    Check -.->|nutzt| Cfg
    LLM --> Ende
"""
mermaid(diagram, width=1100)

In [ ]:
SUMMARY_THRESHOLD = 8
KEEP_RECENT = 4

summary_sessions = {}  # thread_id -> list[BaseMessage]  (aktuelle Nachrichten)
summaries        = {}  # thread_id -> str                 (Zusammenfassung als Text)


def summarize_old_messages(messages: list) -> str:
    prompt = (
        "Extrahiere alle genannten Fakten aus der Konversation als strukturierte Liste.\n"
        "Format: 'Name: ..., Alter: ..., Herkunft: ..., Beruf: ..., Hobbys: ..., Vorlieben: ...'\n"
        "Nur belegte Werte aufführen — fehlende Felder weglassen."
    )
    return llm.invoke([SystemMessage(content=prompt)] + messages).content


def ask_summary(thread_id: str, user_input: str) -> str:
    if thread_id not in summary_sessions:
        summary_sessions[thread_id] = []
        summaries[thread_id]        = ""

    summary_sessions[thread_id].append(HumanMessage(content=user_input))

    # ── 1) Threshold: Brauchen wir eine neue Zusammenfassung? ────────────────
    if len(summary_sessions[thread_id]) > SUMMARY_THRESHOLD:
        to_summarize = summary_sessions[thread_id][:-KEEP_RECENT]

        # ── 2) Rolling Summary: alte Summary + neue Fakten zusammenfassen ────
        # "Rolling" bedeutet: die bisherige Zusammenfassung fließt als Kontext
        # ein, damit keine Informationen aus früheren Runden verloren gehen.
        prefix = (
            [SystemMessage(content=f"Bisherige Zusammenfassung:\n{summaries[thread_id]}")]
            if summaries[thread_id] else []
        )
        summaries[thread_id]        = summarize_old_messages(prefix + to_summarize)
        summary_sessions[thread_id] = summary_sessions[thread_id][-KEEP_RECENT:]

    # ── 3) Kontext für den LLM bauen ─────────────────────────────────────────
    # Reihenfolge: System → Summary (falls vorhanden) → aktuelle Nachrichten
    context = [SystemMessage(content=system_prompt)]
    if summaries[thread_id]:
        context.append(SystemMessage(content=f"Zusammenfassung bisher:\n{summaries[thread_id]}"))
    context += summary_sessions[thread_id]

    response = llm.invoke(context)
    summary_sessions[thread_id].append(response)
    mprint(f"**Mensch:** {user_input}")
    mprint(f"**KI:** {response.content}\n")
    return response.content

In [ ]:
mprint("## Summary Demo (Limit: 8 Nachrichten, behalte 4)")
mprint("---")

for i, user_msg in enumerate([
    "Mein Name ist Max.",
    "Ich bin 30 Jahre alt.",
    "Ich komme aus München.",
    "Ich arbeite als Data Scientist.",
    "Mein Hobby ist Fotografie.",
    "Ich mag italienisches Essen.",
], 1):
    ask_summary("sum_test", user_msg)
    n_msgs = len(summary_sessions["sum_test"])
    has_sum = bool(summaries.get("sum_test"))
    mprint(f"  → summary_sessions['sum_test']: {n_msgs} Listenelemente · summaries: {'ja' if has_sum else 'nein'}")

In [ ]:
# Erinnerungstest
ask_summary("sum_test", "Fasse zusammen: Name, Alter, Herkunft?")

n_msgs = len(summary_sessions["sum_test"])
mprint(f"\n**summary_sessions['sum_test']**: {n_msgs} Listenelemente (aktuelle Nachrichten)")
if summaries.get("sum_test"):
    mprint(f"**summaries['sum_test']:**\n{summaries['sum_test']}")
mprint("✅ Der Bot erinnert sich via Rolling Summary an alte Infos!")

# 4 | Persistentes Memory (SQLite direkt)
---

<p><font color='black' size="5">
4.1 | SQLite-Speicher
</font></p>

---

**Problem:** `sessions = {}` verliert alle Daten beim Neustart der Laufzeitumgebung.

**Lösung:** **SQLite direkt** — das Python-Standardmodul `sqlite3` speichert den Chat-Verlauf persistent in einer Datei. Kein Framework erforderlich.

```python
import sqlite3
conn = sqlite3.connect("./chat_memory.db", check_same_thread=False)
```

**Vorteile:**
- Daten bleiben nach Neustart erhalten
- Vollständige Kontrolle über Schema und Abfragen
- Kein Framework-Overhead
- Multi-Thread-sicher mit `check_same_thread=False`

In [19]:
#@markdown Funktionsaufrufe SQLite direkt
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart LR
    classDef loopStyle fill:#e1f5fe,stroke:#039be5,stroke-width:2px;
    classDef dictStyle fill:#fff3e0,stroke:#fb8c00,stroke-width:2px;
    classDef saveStyle fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;

    subgraph UI ["1. Input / Output"]
        Start["`ask_persistent(conn, thread_id, user_input)`"]:::loopStyle
        Ende["`mprint() / Ausgabe`"]:::loopStyle
    end
    style UI fill:#f0f9ff,stroke:#039be5,stroke-width:1px

    subgraph Engine ["2. SQLite-Logik"]
        direction TB
        Load["`1. load_history(conn, thread_id)<br/>SELECT FROM messages`"]:::dictStyle
        LLM["`2. llm.invoke(context)`"]:::dictStyle
        Save["`3. save_message(conn, ...)<br/>INSERT INTO messages`"]:::dictStyle
        Load --> LLM --> Save
    end
    style Engine fill:#fff9f2,stroke:#fb8c00,stroke-width:1px

    subgraph Storage ["3. SQLite-Datenbank"]
        DB["`chat_memory_m07a.db<br/>Tabelle: messages<br/>thread_id, role, content`"]:::saveStyle
    end
    style Storage fill:#faf5ff,stroke:#8e24aa,stroke-width:1px

    Start --> Load
    Load <--> DB
    Save --> DB
    Save --> Ende
"""
mermaid(diagram, width=1000)

In [ ]:
import sqlite3, os

DB_PATH = "./chat_memory_m07a.db"


def init_db(path: str) -> sqlite3.Connection:
    # check_same_thread=False: erlaubt Zugriff aus dem Notebook-Hauptthread.
    # Notwendig in Colab/Jupyter, da intern mehrere Threads laufen können.
    conn = sqlite3.connect(path, check_same_thread=False)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id         INTEGER PRIMARY KEY AUTOINCREMENT,
            thread_id  TEXT    NOT NULL,
            role       TEXT    NOT NULL,
            content    TEXT    NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    return conn


def save_message(conn: sqlite3.Connection, thread_id: str, role: str, content: str):
    conn.execute(
        "INSERT INTO messages (thread_id, role, content) VALUES (?, ?, ?)",
        (thread_id, role, content)
    )
    conn.commit()


def load_history(conn: sqlite3.Connection, thread_id: str) -> list:
    cursor = conn.execute(
        "SELECT role, content FROM messages WHERE thread_id = ? ORDER BY id",
        (thread_id,)
    )
    messages = []
    for role, content in cursor.fetchall():
        msg = HumanMessage(content=content) if role == "human" else AIMessage(content=content)
        messages.append(msg)
    return messages


def ask_persistent(conn: sqlite3.Connection, thread_id: str, user_input: str) -> str:
    history  = load_history(conn, thread_id)
    history.append(HumanMessage(content=user_input))
    context  = [SystemMessage(content=system_prompt)] + history
    response = llm.invoke(context)
    save_message(conn, thread_id, "human", user_input)
    save_message(conn, thread_id, "ai",    response.content)
    mprint(f"**[{thread_id}] Mensch:** {user_input}")
    mprint(f"**[{thread_id}] KI:** {response.content}\n")
    return response.content


conn = init_db(DB_PATH)
print(f"DB initialisiert: {DB_PATH} ✅")

In [21]:
mprint("## SQLite Demo")
mprint("---")

for thread_id, user_input in [
    ("max",  "Hallo! Ich bin Max aus München."),
    ("max",  "Ich mag Python-Programmierung."),
    ("emma", "Hi! Ich bin Emma aus Berlin."),
    ("max",  "Woher komme ich und was mag ich?"),
]:
    ask_persistent(conn, thread_id, user_input)

mprint("### Gespeicherte Threads")
for tid in ["max", "emma"]:
    hist = load_history(conn, tid)
    mprint(f"- **{tid}**: {len(hist)} Nachrichten")

## SQLite Demo

---

**[max] Mensch:** Hallo! Ich bin Max aus München.

**[max] KI:** Hallo Max! 👋 Schön, dass du da bist — Grüß dich aus der KI-Cloud!  

Wie kann ich dir helfen? Eher was aus München (Biergarten, Wetter, Sehenswürdigkeiten) oder was ganz anderes? 😄


**[max] Mensch:** Ich mag Python-Programmierung.

...

**[max] Mensch:** Woher komme ich und was mag ich?

**[max] KI:** Du hast mir schon zwei Hinweise gegeben, Max:  
- **Woher du kommst:** Du bist **aus München**.  
- **Was du magst:** Du magst **Python-Programmierung**.

Alles klar—aber mal ehrlich: Der nächste “Fakten-Plot-Twist” fehlt noch. 😄  
Was magst du außerdem (z.B. Musik, Gaming, Fußball, Essen, Tech-Themen)? Dann sag ich’s dir in einem hübschen kleinen Steckbrief zusammen.


### Gespeicherte Threads

- **max**: 6 Nachrichten

- **emma**: 2 Nachrichten

In [22]:
# Persistenz-Test: neue Verbindung zur gleichen Datei (simuliert Neustart)
conn2 = sqlite3.connect(DB_PATH, check_same_thread=False)
ask_persistent(conn2, "max", "Was haben wir bisher besprochen?")
mprint("\n✅ **Ergebnis:** Die Historie überlebt den Neustart!")

**[max] Mensch:** Was haben wir bisher besprochen?

**[max] KI:** Bisher haben wir besprochen:

1. Du sagtest: **Hallo**, du bist **Max aus München**.  
2. Ich habe gefragt, wie ich dir helfen kann (München oder etwas anderes).  
3. Du sagtest: **Du magst Python-Programmierung**.  
4. Ich habe nachgefragt, woran du gerade arbeitest und was du als Nächstes willst.  
5. Danach hast du gefragt: **„Woher komme ich und was mag ich?“** – ich habe geantwortet: **aus München** und **Python**.  

Wenn du willst, können wir daraus auch gleich einen kleinen Überblick/“Profil” machen oder direkt zu deinem Python-Thema springen. 😄



✅ **Ergebnis:** Die Historie überlebt den Neustart!

<p><font color='black' size="5">
4.2 | Mehrere Threads
</font></p>

---

In [23]:
def chat_sqlite(thread_id: str, user_input: str) -> str:
    return ask_persistent(conn, thread_id, user_input)

def show_history(thread_id: str):
    msgs = load_history(conn, thread_id)
    mprint(f"### Thread: {thread_id} ({len(msgs)} Nachrichten)\n---")
    for i, msg in enumerate(msgs, 1):
        role = "Mensch" if msg.type == "human" else "KI"
        mprint(f"{i}. **{role}:** {msg.content}")

In [24]:
mprint("## Multi-Thread Demo")
mprint("---")

chat_sqlite("alice", "Hallo! Ich bin Alice und lerne Machine Learning.")
chat_sqlite("bob",   "Hi! Ich bin Bob aus Hamburg.")
chat_sqlite("alice", "Was ist mein Lernthema?")
chat_sqlite("bob",   "Woher komme ich?")

## Multi-Thread Demo

---

**[alice] Mensch:** Hallo! Ich bin Alice und lerne Machine Learning.

**[alice] KI:** Hallo Alice! 👋😊 Willkommen im Club der Maschinenlernenden—wo Modelle träumen und wir Menschen sie dabei mit Daten und Geduld füttern.

Was lernst du gerade konkret: eher **Grundlagen** (z.B. Regression/Klassifikation), **neuronale Netze**, **Training/Overfitting**, oder schon **konkrete Projekte**?  
Und auf welche Sprache/Tools setzt du: **Python + PyTorch**, **TensorFlow**, **scikit-learn**…?


**[bob] Mensch:** Hi! Ich bin Bob aus Hamburg.

...

**[alice] Mensch:** Was ist mein Lernthema?

**[alice] KI:** Dein Lernthema ist **Machine Learning** (bzw. “Maschinelles Lernen”).

Wenn du willst, können wir es auch genauer zuschneiden:  
Meinst du eher **lernen, wie ML funktioniert** (z.B. Regression/Klassifikation, Train/Test, Overfitting), oder **ein konkretes Modell bauen** (z.B. Bild-/Text-/Tabellendaten)?


**[bob] Mensch:** Woher komme ich?

**[bob] KI:** Du sagst doch: **Du bist Bob aus Hamburg**. 😄  
Also kommst du aus **Hamburg**—der Hafenstadt mit dem besten Wetter *direkt nach dem Regen* (zumindest gefühlt).


'Du sagst doch: **Du bist Bob aus Hamburg**. 😄  \nAlso kommst du aus **Hamburg**—der Hafenstadt mit dem besten Wetter *direkt nach dem Regen* (zumindest gefühlt).'

In [25]:
mprint("### Alle Threads:")
mprint("---")
for tid in ["alice", "bob", "max", "emma"]:
    hist = load_history(conn, tid)
    if hist:
        mprint(f"- **{tid}**: {len(hist)} Nachrichten")

show_history("alice")

### Alle Threads:

---

- **alice**: 4 Nachrichten

- **bob**: 4 Nachrichten

- **max**: 8 Nachrichten

...

### Thread: alice (4 Nachrichten)
---

1. **Mensch:** Hallo! Ich bin Alice und lerne Machine Learning.

2. **KI:** Hallo Alice! 👋😊 Willkommen im Club der Maschinenlernenden—wo Modelle träumen und wir Menschen sie dabei mit Daten und Geduld füttern.

Was lernst du gerade konkret: eher **Grundlagen** (z.B. Regression/Klassifikation), **neuronale Netze**, **Training/Overfitting**, oder schon **konkrete Projekte**?  
Und auf welche Sprache/Tools setzt du: **Python + PyTorch**, **TensorFlow**, **scikit-learn**…?

3. **Mensch:** Was ist mein Lernthema?

4. **KI:** Dein Lernthema ist **Machine Learning** (bzw. “Maschinelles Lernen”).

Wenn du willst, können wir es auch genauer zuschneiden:  
Meinst du eher **lernen, wie ML funktioniert** (z.B. Regression/Klassifikation, Train/Test, Overfitting), oder **ein konkretes Modell bauen** (z.B. Bild-/Text-/Tabellendaten)?

In [26]:
# Neue Verbindung zur gleichen Datei (simuliert Neustart)
conn_restart = sqlite3.connect(DB_PATH, check_same_thread=False)
ask_persistent(conn_restart, "alice", "Was lerne ich gerade?")
mprint("\n✅ **Ergebnis:** Kontext aus früherer Session korrekt geladen!")

**[alice] Mensch:** Was lerne ich gerade?

**[alice] KI:** Du lernst gerade **Machine Learning** – also, wie man aus Daten Vorhersagen oder Entscheidungen ableitet, mit Modellen, die aus Beispielen “lernen”. 🤖📚

Wenn du magst, sag mir kurz:
1) Was steht bei dir diese Woche auf dem Plan? (z.B. Regression, Klassifikation, Neuronale Netze)  
2) Nutzt du dafür Python mit scikit-learn/PyTorch/TensorFlow?  
Dann kann ich dir dein Lernthema ganz konkret benennen.



✅ **Ergebnis:** Kontext aus früherer Session korrekt geladen!

In [27]:
def delete_db():
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Gelöscht: {DB_PATH}")

# Auskommentiert, um versehentliches Löschen zu verhindern:
# delete_db()

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> Wichtig ist nur: Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, GenAI-Apps selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

**Grundlagen**

Teste **Trimming**-Limits in einem Chatgespräch und beobachte, ab wann der Kontext verloren geht (mind. 5 Nachrichten).

**✅ Erledigt wenn:** Ab einer bestimmten Nachrichtenanzahl gibt das Modell an, die frühere Frage nicht mehr zu kennen — der Grenzwert ist dokumentiert.

In [28]:
# Grundlagen: Trimming-Limits testen
# Startpunkt: ask_trimmed() aus Abschnitt 3.2

# Ablauf: Chatgespräch mit mindestens 5 Nachrichten führen
# Nach jeder Runde fragen: "Was war meine erste Frage?"
# Beobachten: Ab wann ist die Antwort "Das weiß ich nicht mehr"
# Grenzwert dokumentieren
# ...

**Aufbau**

Verbessere den Summary-Prompt oder baue einen interaktiven CLI-Chatbot mit Threads, der Kontext über mehrere Nachrichten hält.

**✅ Erledigt wenn:** Der Chatbot hält Kontext über mindestens fünf aufeinander aufbauende Fragen — eine Rückfrage auf die erste Frage wird korrekt beantwortet.

In [29]:
# Aufbau: CLI-Chatbot mit Summary oder Threads
# Startpunkt: ask_summary() aus Abschnitt 3.3

# Option A: Summary-Prompt verbessern
# Option B: Interaktiver CLI-Chatbot mit while-Schleife und Thread-ID

# Test: 5 aufeinander aufbauende Fragen stellen
# Dann: "Was war meine erste Frage?" → muss korrekt beantwortet werden
# ...

**Vertiefung**

Kombiniere Trimming, Summary und SQLite zu einem Hybrid-Memory-System für persistente Chatsitzungen.

**✅ Erledigt wenn:** Der Chatbot liest beim Start gespeicherte Sitzungen aus der SQLite-Datenbank — Kontext aus früheren Sitzungen fließt in die Antwort ein.

In [30]:
# Vertiefung: Hybrid-Memory-System (Trimming + Summary + SQLite)
# Startpunkt: SQLite-Beispiel aus Abschnitt 4

# 1. Trimming: kurzen Kontext im Arbeitsspeicher halten
# 2. Summary: ältere Nachrichten zusammenfassen und in DB speichern
# 3. SQLite: Sitzungen persistent speichern und beim Start laden

# Test: Colab neu starten → gespeicherter Kontext muss wiederhergestellt werden
# ...

# B | Datenbank auslesen
---

Threads aus `conn` auslesen — nützlich für Debugging, Analyse oder Export.

In [31]:
def read_all_threads(thread_ids: list, connection=None):
    if connection is None:
        connection = conn

    mprint(f"### Alle Threads ({len(thread_ids)} gesamt)\n---")
    for tid in thread_ids:
        msgs = load_history(connection, tid)
        if not msgs:
            continue
        mprint(f"#### Thread: {tid} ({len(msgs)} Nachrichten)")
        for i, msg in enumerate(msgs, 1):
            role  = "Mensch" if msg.type == "human" else "KI"
            short = msg.content[:100] + "..." if len(msg.content) > 100 else msg.content
            mprint(f"{i}. **{role}:** {short}")
        mprint("")

In [32]:
read_all_threads(["max", "emma", "alice", "bob"])

### Alle Threads (4 gesamt)
---

#### Thread: max (8 Nachrichten)

1. **Mensch:** Hallo! Ich bin Max aus München.

2. **KI:** Hallo Max! 👋 Schön, dass du da bist — Grüß dich aus der KI-Cloud!  

Wie kann ich dir helfen? Eher w...

3. **Mensch:** Ich mag Python-Programmierung.

...

#### Thread: bob (4 Nachrichten)

1. **Mensch:** Hi! Ich bin Bob aus Hamburg.

2. **KI:** Hey Bob! 👋 Ich bin’s—dein KI-Kumpel.  

Hamburg, nice! Wie läuft’s bei dir heute: eher **Reeperbahn-...

3. **Mensch:** Woher komme ich?

4. **KI:** Du sagst doch: **Du bist Bob aus Hamburg**. 😄  
Also kommst du aus **Hamburg**—der Hafenstadt mit de...

In [33]:
import json

def export_thread_to_json(thread_id: str, connection=None) -> dict:
    if connection is None:
        connection = conn
    msgs = load_history(connection, thread_id)
    return {
        "thread_id": thread_id,
        "messages":  [{"role": m.type, "content": m.content} for m in msgs]
    }

thread_data = export_thread_to_json("max")
print(json.dumps(thread_data, indent=2, ensure_ascii=False))

{
  "thread_id": "max",
  "messages": [
    {
      "role": "human",
      "content": "Hallo! Ich bin Max aus München."
    },
    {
      "role": "ai",
      "content": "Hallo Max! 👋 Schön, dass du da bist — Grüß dich aus der KI-Cloud!  \n\nWie kann ich dir helfen? Eher was aus München (Biergarten, Wetter, Sehenswürdigkeiten) oder was ganz anderes? 😄"
    },
    {
      "role": "human",
      "content": "Ich mag Python-Programmierung."
    },
    {
      "role": "ai",
      "content": "Nice, Max! Python ist wirklich ein Allrounder — wie ein Schweizer Taschenmesser, nur mit weniger Schraubenziehern und mehr “why not?”. 😄🐍\n\nWoran arbeitest du gerade (z.B. Datenanalyse, Web, Automatisierung, Bots, Scripts)?  \nUnd was möchtest du als Nächstes: **Hilfe bei einem Problem**, **eine Idee ausarbeiten**, oder **best practices / Lernplan**?"
    },
    {
      "role": "human",
      "content": "Woher komme ich und was mag ich?"
    },
    {
      "role": "ai",
      "content": "Du hast mir sc

In [34]:
def delete_db_file():
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Gelöscht: {DB_PATH}")

# Auskommentiert, um versehentliches Löschen zu verhindern:
# delete_db_file()